# Migrate a MySQL RDS snapshot to S3 Parquet using AWS DMS with row-count validation

The platform team is decommissioning the legacy `payments-oltp` MySQL server in 30 days. Five years of order history has to land in the data lake as Parquet before the source goes dark, and finance wants a row-count audit trail before they sign off on the migration. You have one session to build the DMS pipeline and prove every row made it across.

Your job is the DMS replication instance plus full-load task that moves MySQL `orders` rows to S3 Parquet, plus a row-count assertion that compares source `SELECT count(*)` against the Parquet file row totals. RDS MySQL `db.t4g.micro` on the source side, S3 Parquet on the target side, `dms.t3.micro` replication instance in between.

The four deliverables map to four checkpoints:
1. RDS MySQL `available` with a seeded `orders` table holding at least 10000 rows.
2. DMS replication instance `available`; source MySQL endpoint and S3 target endpoint exist and both pass `test_connection`.
3. DMS full-load task completes (`Status == "stopped"` with `StopReason` indicating load complete) and statistics report non-zero `FullLoadRows`.
4. Parquet file row count under `s3://{BUCKET}/orders/` matches the MySQL `SELECT count(*)` (within plus or minus 1 row).

**Time.** About 65 minutes hands-on. Most of the wall-clock time is the RDS create wait (4 to 8 minutes), the DMS instance create wait (8 to 12 minutes), and the full-load task run (1 to 3 minutes on the seeded dataset). Budget 90 minutes if your first DMS endpoint connection fails the TLS check and you need to retry.

**Cost.** This lab costs about 5 to 30 cents per session. The two hourly meters are RDS `db.t4g.micro` ($0.017/hour, free tier if eligible) and DMS `dms.t3.micro` ($0.018/hour, billed continuously from launch until deletion **even while the task is stopped**). Cleanup deletes the DMS replication task first, the DMS replication instance second (this is what actually stops the bill), the RDS instance third with `SkipFinalSnapshot=True`, then standard resources. A 2-hour EventBridge plus Lambda safety net auto-deletes the three critical resources if the kernel dies before cleanup runs.

This lab maps to AWS DEA-C01 Domain 3: Data Operations and Support (22% of exam weight).


In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.7.0


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12. Lab 09 is a critical-resource lab; two hourly meters
# (RDS db.t4g.micro and DMS dms.t3.micro), one of which (DMS) bills
# continuously from launch until deletion even while the task is stopped.

import atexit
import getpass
import io
import json
import random
import string
import sys
import time
import urllib.request
import zipfile
from datetime import datetime, timedelta, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "dms-database-migration"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[8].slug exactly
REGION = "us-east-1"  # all DEA-C01 labs run in us-east-1 per LAB-CREATION-BLUEPRINT section 15

# Resource names. Bucket name appends the account ID for global uniqueness.
RDS_ID = f"labrun-{LAB_ID}-rds"
SG_RDS_NAME = f"labrun-{LAB_ID}-sg-rds"
SG_DMS_NAME = f"labrun-{LAB_ID}-sg-dms"
SUBNET_GROUP_NAME = f"labrun-{LAB_ID}-subnet-group"
DMS_S3_ROLE_NAME = f"labrun-{LAB_ID}-s3-role"
DMS_INSTANCE_ID = f"labrun-{LAB_ID}-instance"
SOURCE_ENDPOINT_ID = f"labrun-{LAB_ID}-source"
TARGET_ENDPOINT_ID = f"labrun-{LAB_ID}-target"
TASK_ID = f"labrun-{LAB_ID}-task"
SAFETY_NET_LAMBDA_NAME = f"labrun-{LAB_ID}-safety-net"
SAFETY_NET_RULE_NAME = f"labrun-{LAB_ID}-safety-net-rule"
SAFETY_NET_ROLE_NAME = f"labrun-{LAB_ID}-safety-net-role"
BUCKET_NAME = None  # resolved after STS once the account ID is known

# RDS shape. db.t4g.micro is the cheapest current-gen MySQL shape and is the
# free-tier-eligible option for the first 12 months of an AWS account.
DB_ENGINE = "mysql"
DB_INSTANCE_CLASS = "db.t4g.micro"
DB_NAME = "labrun"
DB_USER = "labrun_admin"
DB_PORT = 3306

# Generated admin password. RDS MySQL requires 8 to 41 chars; no @ or /.
_pw_chars = string.ascii_letters + string.digits
_rng = random.Random(0xC0FFEE)
DB_PASSWORD = "Lab" + "".join(_rng.choice(_pw_chars) for _ in range(20)) + "9"

# DMS shape. dms.t3.micro is the cheapest replication instance class.
DMS_INSTANCE_CLASS = "dms.t3.micro"
DMS_ENGINE_VERSION = "3.5.2"
DMS_ALLOCATED_STORAGE = 20  # GB

# Seed config. Deterministic so the source row count and the Parquet target
# row count line up exactly on a clean run. Checkpoint 4 asserts equality
# within plus or minus 1 row.
SEED = 42
ORDERS_ROWS = 10_000

# Sample-dataset fallback constants. If the student is past their 12-month
# free-tier window for RDS, the lab refuses to provision RDS and runs
# Checkpoints 3 and 4 against a deterministic Parquet sample seeded into
# s3://{BUCKET}/orders/. SAMPLE_ROWS_EXPECTED is the row total Checkpoint
# 4 compares the read Parquet row sum against in fallback mode.
SAMPLE_ROWS_EXPECTED = ORDERS_ROWS
USE_RDS = True  # resolved by the prompt in the setup cell

# Safety-net schedule. The three critical resources (DMS task, DMS instance,
# RDS instance) auto-destroy 2 hours after setup runs per RESOURCE-SAFETY-SPEC
# Section 3 Lab 9 mitigation. The EventBridge schedule expression is computed
# at setup time relative to wall clock.
SAFETY_NET_HOURS = 2

# Module-level caches populated by tasks for the checkpoints to read.
SOURCE_ENDPOINT_ARN: str | None = None
TARGET_ENDPOINT_ARN: str | None = None
DMS_INSTANCE_ARN: str | None = None
TASK_ARN: str | None = None
SOURCE_ROW_COUNT: int | None = None  # set by Task 4 from MySQL or sample
PARQUET_ROW_COUNT: int | None = None  # set by Task 4 from Parquet reads

# DMS table-mapping JSON: include the labrun.orders table only.
TABLE_MAPPINGS = {
    "rules": [
        {
            "rule-type": "selection",
            "rule-id": "1",
            "rule-name": "1",
            "object-locator": {
                "schema-name": "labrun",
                "table-name": "orders",
            },
            "rule-action": "include",
        }
    ],
}


In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# surface the two confirmation prompts required by RESOURCE-SAFETY-SPEC
# Section 3 Lab 9 mitigation (DMS-bills-while-stopped + RDS free-tier
# eligibility), and finally call register(). This cell must succeed before
# the manifest cell creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

# Validate credentials against AWS via STS GetCallerIdentity. Fail fast with a
# clear message rather than waiting for the first RDS or DMS call to error.
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

# Resolve the bucket name now that we know the account ID. S3 bucket names
# must be globally unique.
BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Confirmation prompt per RESOURCE-SAFETY-SPEC Section 3 Lab 9 mitigation.
# DMS bills while the task is stopped; the student must acknowledge the
# critical-teardown semantics before any meter starts.
confirm = input(
    "DMS bills continuously from launch until deletion, even while tasks are stopped.\n"
    "The critical teardown is the DMS replication instance, not the task.\n"
    "Type 'I understand' to proceed: "
)
if confirm.strip() != "I understand":
    print("Setup aborted. Type 'I understand' exactly to proceed.")
    raise SystemExit(1)
print("Acknowledged. The 2-hour safety net will catch a forgotten DMS instance.")

# Free-tier eligibility prompt. The Billing API exposes free-tier usage only
# to consolidated billing accounts, so this is a manual self-check. If the
# student is past their first 12 months, the lab refuses to provision RDS
# and runs Checkpoints 3 and 4 against the seeded sample Parquet dataset.
within_free_tier = input(
    "Are you within the first 12 months of your AWS account creation? [y/N]: "
).strip().lower()
USE_RDS = within_free_tier == "y"
if not USE_RDS:
    print(
        "Lab will skip RDS provisioning and run Checkpoints 3 and 4 "
        "against the seeded sample dataset."
    )
else:
    print("RDS provisioning path selected. db.t4g.micro is free-tier eligible.")

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
#
# Per RESOURCE-SAFETY-SPEC Rule 2 + Section 3 Lab 9 mitigation, the three
# hourly-billed (or safety-net) resource pairs tear down FIRST in this
# exact order:
#   1. DMS replication task (must be stopped + deleted before the instance,
#      otherwise delete-instance blocks).
#   2. DMS replication instance (highest billing-while-stopped exposure;
#      this is the deletion that actually stops the DMS meter).
#   3. RDS instance (skip-final-snapshot, delete-automated-backups).
#   4. Safety-net EventBridge rule + Lambda (no point keeping the seatbelt
#      after the cluster, instance, and task are gone).
# Then standard resources (DMS endpoints, subnet group, IAM roles, security
# groups, S3 bucket). The manifest below lists every resource the lab
# creates in strict reverse-creation, critical-first order so the atexit
# handler tears down the meters even if the kernel dies during the
# create_replication_instance waiter.
#
# Per RESOURCE-SAFETY-SPEC Rule 4, the orphan scan blocks execution if any
# tagged resources from a prior session exist (not just print a warning).
#
# labrun-checks 0.3.0 lacks adapters for dms_replication_task,
# dms_replication_instance, rds_instance, dms_endpoint, dms_subnet_group,
# eventbridge_rule, lambda_function, and security_group; inline _LabAdapter
# adds them. The new resource types are still declared declaratively here
# so the canonical summary, atexit handler, and orphan scan all see them.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="dms_replication_task",
        id=TASK_ID,
        region=REGION,
        cli_delete_command=(
            f"aws dms stop-replication-task --replication-task-arn $TASK_ARN && "
            f"aws dms delete-replication-task --replication-task-arn $TASK_ARN"
        ),
        critical=True,
    ),
    CleanupResource(
        type="dms_replication_instance",
        id=DMS_INSTANCE_ID,
        region=REGION,
        cli_delete_command=(
            f"aws dms delete-replication-instance "
            f"--replication-instance-arn $DMS_INSTANCE_ARN"
        ),
        critical=True,
    ),
    CleanupResource(
        type="rds_instance",
        id=RDS_ID,
        region=REGION,
        cli_delete_command=(
            f"aws rds delete-db-instance --db-instance-identifier {RDS_ID} "
            f"--skip-final-snapshot --delete-automated-backups"
        ),
        critical=True,
    ),
    CleanupResource(
        type="eventbridge_rule",
        id=SAFETY_NET_RULE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws events remove-targets --rule {SAFETY_NET_RULE_NAME} --ids 1 && "
            f"aws events delete-rule --name {SAFETY_NET_RULE_NAME}"
        ),
        critical=True,
    ),
    CleanupResource(
        type="lambda_function",
        id=SAFETY_NET_LAMBDA_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws lambda delete-function --function-name {SAFETY_NET_LAMBDA_NAME}"
        ),
        critical=True,
    ),
    CleanupResource(
        type="dms_endpoint",
        id=TARGET_ENDPOINT_ID,
        region=REGION,
        cli_delete_command=(
            f"aws dms delete-endpoint --endpoint-arn $TARGET_ENDPOINT_ARN"
        ),
    ),
    CleanupResource(
        type="dms_endpoint",
        id=SOURCE_ENDPOINT_ID,
        region=REGION,
        cli_delete_command=(
            f"aws dms delete-endpoint --endpoint-arn $SOURCE_ENDPOINT_ARN"
        ),
    ),
    CleanupResource(
        type="dms_subnet_group",
        id=SUBNET_GROUP_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws dms delete-replication-subnet-group "
            f"--replication-subnet-group-identifier {SUBNET_GROUP_NAME}"
        ),
    ),
    CleanupResource(
        type="iam_role",
        id=SAFETY_NET_ROLE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws iam delete-role --role-name {SAFETY_NET_ROLE_NAME}"
        ),
    ),
    CleanupResource(
        type="iam_role",
        id=DMS_S3_ROLE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws iam delete-role --role-name {DMS_S3_ROLE_NAME}"
        ),
    ),
    CleanupResource(
        type="security_group",
        id=SG_DMS_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws ec2 delete-security-group --group-name {SG_DMS_NAME}"
        ),
    ),
    CleanupResource(
        type="security_group",
        id=SG_RDS_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws ec2 delete-security-group --group-name {SG_RDS_NAME}"
        ),
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    The dedicated cleanup cell is the authoritative entry point; this is
    the safety net for kernel crashes during the create_replication_instance
    waiter or any other long-running step. Errors are swallowed because
    atexit handlers must not raise.
    """
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter()) if "_LabAdapter" in globals() else run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for leftover
    state with the lab's tag and surface the cleanup command before creating
    any new resources. Critical for Lab 09 because a leftover DMS replication
    instance bills at $0.018 per hour continuously even with no task running.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources or,")
    print("worse, a second DMS replication instance billing in parallel.")
    print("Run the cleanup cell at the bottom of this notebook first, or remove")
    print("them manually with the AWS CLI commands the cleanup cell prints,")
    print("then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Task 1: Provision the RDS MySQL source and seed the orders table (or take the sample-data fallback)

Two paths here, branched on the free-tier eligibility prompt you answered in setup.

**Path A: `USE_RDS == True`.** You create the bucket, the lab security groups, an RDS `db.t4g.micro` MySQL instance in the default VPC's public subnet, and seed the `orders` table with `ORDERS_ROWS` rows. The seed runs through `pymysql` (lazy-installed if missing). The RDS instance is tagged `labrun:lab-slug` at creation and pre-declared in `CLEANUP_MANIFEST` with `critical=True` so the atexit handler catches a kernel crash during the create-DB waiter.

**Path B: `USE_RDS == False`.** You skip RDS provisioning entirely, create the bucket and the security groups, and seed the read-only Parquet sample into `s3://{BUCKET}/orders/` directly. Checkpoint 1 confirms the sample-data flag is set and the sample dataset exists. Checkpoints 3 and 4 run against the seeded Parquet without a DMS round trip. This path costs less than $0.01 because the only resources that move money are S3 PUT requests.

Either way, the bucket goes in first because Task 2 will write the DMS S3 target endpoint against it. The two security groups go in next (one for RDS, one for the DMS replication instance) so the DMS-to-RDS network path is cleared before DMS comes up in Task 2. The lab uses two SGs and two ingress rules: caller IP plus DMS SG can both reach RDS on port 3306.

The 2-hour EventBridge plus Lambda safety net is created in Task 2 alongside the DMS replication instance. It is created **before** the DMS create call so the seatbelt is in place when the meter starts.

### Checkpoint 1 validates

- `USE_RDS == True`: `rds.describe_db_instances(DBInstanceIdentifier=RDS_ID)` returns `available`; `Engine == "mysql"`; `PubliclyAccessible is True`; `pymysql` can connect via the public endpoint and `SELECT count(*) FROM orders` returns at least `ORDERS_ROWS` rows.
- `USE_RDS == False`: the global `USE_RDS` flag is `False`, the sample Parquet under `s3://{BUCKET}/orders/` exists, and the manifest carries the bucket entry.


In [ ]:
# NBVAL_SKIP
# Task 1: Create the S3 bucket, the two security groups, and either provision
# the RDS MySQL instance + seed orders (Path A) or seed the sample Parquet
# directly (Path B). The 2-hour safety-net pair (EventBridge rule + Lambda)
# is created in Task 2 because it depends on the DMS instance ID being
# resolved.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
ec2 = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
rds = boto3.client(
    "rds",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# 1. Bucket. us-east-1 rejects LocationConstraint; other regions require it.
# YOUR CODE: s3.create_bucket(Bucket=BUCKET_NAME).
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

# 2. Default VPC + subnet discovery (used by both SGs and by Task 2's
# DMS subnet group).
default_vpc = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])
default_vpc_id = default_vpc["Vpcs"][0]["VpcId"]
subnet_resp = ec2.describe_subnets(
    Filters=[{"Name": "vpc-id", "Values": [default_vpc_id]}],
)
subnet_ids = [s["SubnetId"] for s in subnet_resp.get("Subnets", [])][:3]
if len(subnet_ids) < 2:
    print("Default VPC has fewer than 2 subnets; DMS subnet group needs >= 2.")
    raise SystemExit(1)
print(f"Default VPC: {default_vpc_id}; using subnets {subnet_ids}")

# 3. Security groups. Two of them: one for RDS, one for the DMS instance.
# The RDS SG allows ingress from BOTH the caller IP (for the seed step) and
# the DMS SG (for the DMS source endpoint connection during the migration).
my_ip = urllib.request.urlopen("https://checkip.amazonaws.com").read().decode().strip()
my_cidr = f"{my_ip}/32"

dms_sg_resp = ec2.create_security_group(
    GroupName=SG_DMS_NAME,
    Description=f"DMS replication instance SG for {LAB_ID}",
    VpcId=default_vpc_id,
    TagSpecifications=[{
        "ResourceType": "security-group",
        "Tags": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    }],
)
dms_sg_id = dms_sg_resp["GroupId"]
print(f"DMS SG created: {dms_sg_id} ({SG_DMS_NAME})")

rds_sg_resp = ec2.create_security_group(
    GroupName=SG_RDS_NAME,
    Description=f"RDS access SG for {LAB_ID}, locked to caller IP plus DMS SG",
    VpcId=default_vpc_id,
    TagSpecifications=[{
        "ResourceType": "security-group",
        "Tags": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    }],
)
rds_sg_id = rds_sg_resp["GroupId"]
# Allow caller IP (for the seed) and the DMS SG (for the migration).
ec2.authorize_security_group_ingress(
    GroupId=rds_sg_id,
    IpPermissions=[
        {
            "IpProtocol": "tcp",
            "FromPort": DB_PORT,
            "ToPort": DB_PORT,
            "IpRanges": [{"CidrIp": my_cidr, "Description": "labrun lab caller IP"}],
        },
        {
            "IpProtocol": "tcp",
            "FromPort": DB_PORT,
            "ToPort": DB_PORT,
            "UserIdGroupPairs": [
                {"GroupId": dms_sg_id, "Description": "DMS replication instance"}
            ],
        },
    ],
)
print(f"RDS SG created: {rds_sg_id} ({SG_RDS_NAME}); ingress 3306 from {my_cidr} and {dms_sg_id}")

# Cache for Task 2 + checkpoints.
SUBNET_IDS = subnet_ids
DMS_SG_ID = dms_sg_id
RDS_SG_ID = rds_sg_id
DEFAULT_VPC_ID = default_vpc_id

if USE_RDS:
    # Path A: provision RDS, wait, seed.
    # YOUR CODE: rds.create_db_instance(
    #     DBInstanceIdentifier=RDS_ID,
    #     AllocatedStorage=20,
    #     DBInstanceClass=DB_INSTANCE_CLASS,
    #     Engine=DB_ENGINE,
    #     MasterUsername=DB_USER,
    #     MasterUserPassword=DB_PASSWORD,
    #     DBName=DB_NAME,
    #     VpcSecurityGroupIds=[RDS_SG_ID],
    #     PubliclyAccessible=True,
    #     BackupRetentionPeriod=0,
    #     Port=DB_PORT,
    #     Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    # ).
    print(f"create_db_instance returned for {RDS_ID}; meter started at $0.017/hour (free-tier eligible).")
    print("Waiting for RDS instance to become available (4 to 8 minutes typical)...")
    waiter = rds.get_waiter("db_instance_available")
    waiter.wait(
        DBInstanceIdentifier=RDS_ID,
        WaiterConfig={"Delay": 30, "MaxAttempts": 30},
    )
    desc = rds.describe_db_instances(DBInstanceIdentifier=RDS_ID)
    db_instance = desc["DBInstances"][0]
    RDS_ENDPOINT_HOST = db_instance["Endpoint"]["Address"]
    print(f"RDS available at {RDS_ENDPOINT_HOST}:{DB_PORT}")

    # Seed via pymysql. Lazy-install if not present; the install is about
    # 10 seconds. pymysql.install_as_MySQLdb() registers the package under
    # both names so legacy MySQLdb imports work too.
    try:
        import pymysql  # noqa: F401
    except ImportError:
        print("pymysql not installed; installing now (about 10 seconds)...")
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "pymysql"])
    import pymysql
    pymysql.install_as_MySQLdb()

    conn = pymysql.connect(
        host=RDS_ENDPOINT_HOST,
        port=DB_PORT,
        user=DB_USER,
        password=DB_PASSWORD,
        database=DB_NAME,
        connect_timeout=30,
        ssl={"ssl": {}},  # require TLS (RDS forces it by default on newer engines)
    )
    try:
        with conn.cursor() as cur:
            cur.execute("CREATE DATABASE IF NOT EXISTS labrun;")
            cur.execute("USE labrun;")
            cur.execute("DROP TABLE IF EXISTS orders;")
            cur.execute(
                "CREATE TABLE orders ("
                "  order_id BIGINT PRIMARY KEY, "
                "  customer_id INT, "
                "  amount_cents INT, "
                "  status VARCHAR(16), "
                "  created_at DATETIME"
                ") ENGINE=InnoDB;"
            )
            _seed_rng = random.Random(SEED)
            statuses = ["paid", "refunded", "pending", "failed"]
            batch = []
            for i in range(ORDERS_ROWS):
                batch.append((
                    i + 1,
                    _seed_rng.randint(1, 500),
                    _seed_rng.randint(100, 100_000),
                    _seed_rng.choice(statuses),
                    f"2025-{_seed_rng.randint(1,12):02d}-{_seed_rng.randint(1,28):02d} "
                    f"{_seed_rng.randint(0,23):02d}:{_seed_rng.randint(0,59):02d}:00",
                ))
                if len(batch) >= 1000:
                    cur.executemany(
                        "INSERT INTO orders (order_id, customer_id, amount_cents, status, created_at) "
                        "VALUES (%s, %s, %s, %s, %s)",
                        batch,
                    )
                    batch = []
            if batch:
                cur.executemany(
                    "INSERT INTO orders (order_id, customer_id, amount_cents, status, created_at) "
                    "VALUES (%s, %s, %s, %s, %s)",
                    batch,
                )
            conn.commit()
            cur.execute("SELECT count(*) FROM orders;")
            row = cur.fetchone()
            print(f"Seed complete. SELECT count(*) FROM orders = {row[0]}")
    finally:
        conn.close()
else:
    # Path B: seed the sample Parquet directly into s3://{BUCKET}/orders/.
    # The Parquet bytes match what the DMS S3 target would have written;
    # Checkpoint 3 and Checkpoint 4 both pass against the sample without
    # a DMS round trip.
    print("Sample-data path: skipping RDS provisioning. Writing sample Parquet to S3.")
    try:
        import pyarrow as pa  # noqa: F401
        import pyarrow.parquet as pq  # noqa: F401
    except ImportError:
        print("pyarrow not installed; installing now (about 15 seconds)...")
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "pyarrow"])
    import pyarrow as pa
    import pyarrow.parquet as pq

    _seed_rng = random.Random(SEED)
    statuses = ["paid", "refunded", "pending", "failed"]
    order_ids, customer_ids, amounts, statuses_col, created_ats = [], [], [], [], []
    for i in range(SAMPLE_ROWS_EXPECTED):
        order_ids.append(i + 1)
        customer_ids.append(_seed_rng.randint(1, 500))
        amounts.append(_seed_rng.randint(100, 100_000))
        statuses_col.append(_seed_rng.choice(statuses))
        created_ats.append(
            f"2025-{_seed_rng.randint(1,12):02d}-{_seed_rng.randint(1,28):02d}"
        )
    table = pa.table({
        "order_id": order_ids,
        "customer_id": customer_ids,
        "amount_cents": amounts,
        "status": statuses_col,
        "created_at": created_ats,
    })
    buf = io.BytesIO()
    pq.write_table(table, buf)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key="orders/sample-LOAD00000001.parquet",
        Body=buf.getvalue(),
        Tagging=f"{LAB_TAG_KEY}={LAB_TAG_VALUE}",
    )
    print(f"Sample Parquet seeded: s3://{BUCKET_NAME}/orders/sample-LOAD00000001.parquet "
          f"({SAMPLE_ROWS_EXPECTED} rows)")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: RDS MySQL is available with seeded orders (Path A), or the
# sample-data flag is set and the sample Parquet exists (Path B).


def checkpoint_1(session):
    try:
        if USE_RDS:
            rds_client = boto3.client(
                "rds",
                aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
                aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
                aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
                region_name=REGION,
            )
            try:
                resp = rds_client.describe_db_instances(DBInstanceIdentifier=RDS_ID)
            except ClientError as e:
                if e.response["Error"]["Code"] == "DBInstanceNotFound":
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"RDS instance {RDS_ID!r} does not exist. "
                            f"Run the Task 1 cell to create it."
                        ),
                    )
                return CheckpointResult(status="error", error_reason=str(e))

            dbs = resp.get("DBInstances", [])
            if not dbs:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"describe_db_instances returned no instances for {RDS_ID!r}.",
                )
            db = dbs[0]
            status_ = db.get("DBInstanceStatus", "")
            if status_ != "available":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"DBInstanceStatus is {status_!r}, expected 'available'. "
                        f"Wait for the create_db_instance waiter and re-run."
                    ),
                )
            engine = db.get("Engine", "")
            if engine != DB_ENGINE:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Engine is {engine!r}, expected {DB_ENGINE!r}.",
                )
            if not db.get("PubliclyAccessible", False):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        "PubliclyAccessible is False. The lab uses the default VPC's "
                        "public subnet with IP-locked security groups."
                    ),
                )

            # Connect through pymysql and run SELECT count(*).
            try:
                import pymysql
            except ImportError:
                import subprocess
                subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "pymysql"])
                import pymysql
            endpoint = db["Endpoint"]["Address"]
            try:
                conn = pymysql.connect(
                    host=endpoint, port=DB_PORT,
                    user=DB_USER, password=DB_PASSWORD,
                    database=DB_NAME, connect_timeout=15,
                    ssl={"ssl": {}},
                )
            except Exception as exc:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"pymysql.connect to {endpoint}:{DB_PORT} failed: {exc}. "
                        f"Check that the RDS SG allows ingress from the caller IP."
                    ),
                )
            try:
                with conn.cursor() as cur:
                    cur.execute("SELECT count(*) FROM orders;")
                    row = cur.fetchone()
                    n = int(row[0])
            finally:
                conn.close()
            if n < ORDERS_ROWS:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"orders has {n} rows, expected at least {ORDERS_ROWS}. "
                        f"Re-run the seed step in Task 1."
                    ),
                )
            return CheckpointResult(status="pass")
        else:
            # Path B: sample-data flag must be set and the sample Parquet
            # must exist under s3://{BUCKET}/orders/.
            s3_client = boto3.client(
                "s3",
                aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
                aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
                aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
                region_name=REGION,
            )
            try:
                listing = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix="orders/")
            except ClientError as e:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Could not list s3://{BUCKET_NAME}/orders/: {e}. "
                        f"Re-run Task 1 to seed the sample dataset."
                    ),
                )
            contents = listing.get("Contents", [])
            if not contents:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"No objects under s3://{BUCKET_NAME}/orders/. "
                        f"Re-run Task 1 (sample-data path) to seed the Parquet sample."
                    ),
                )
            return CheckpointResult(
                status="pass",
                error_reason=None,
            )
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint branches on `USE_RDS`. In Path A it walks four properties in order: instance exists, `DBInstanceStatus == "available"`, `Engine == "mysql"`, `PubliclyAccessible is True`, plus a `pymysql.connect` that runs `SELECT count(*) FROM orders` and asserts the row count is at least `ORDERS_ROWS`. The most common miss on the first run is forgetting `PubliclyAccessible=True` on `create_db_instance`; the instance comes up private and the `pymysql.connect` hangs until the 15-second timeout. The next most common miss is forgetting the RDS SG ingress rule for the caller IP; the connection times out the same way but the security group check tells you exactly why. In Path B the checkpoint only confirms the sample Parquet under `s3://{BUCKET}/orders/` exists; the sample-data flag bypasses the live-DB check entirely.

</details>

<details><summary>Hint 2 (direction)</summary>

Five API calls in Path A. `s3.create_bucket(Bucket=BUCKET_NAME)` (no `LocationConstraint` in us-east-1). Two `ec2.create_security_group` calls plus the two `authorize_security_group_ingress` calls (caller IP on RDS SG, plus DMS SG referenced as a `UserIdGroupPairs` entry on RDS SG). `rds.create_db_instance(DBInstanceIdentifier=RDS_ID, AllocatedStorage=20, DBInstanceClass=DB_INSTANCE_CLASS, Engine=DB_ENGINE, MasterUsername=DB_USER, MasterUserPassword=DB_PASSWORD, DBName=DB_NAME, VpcSecurityGroupIds=[RDS_SG_ID], PubliclyAccessible=True, BackupRetentionPeriod=0, Port=DB_PORT, Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])`. `BackupRetentionPeriod=0` keeps automated backups off so delete returns fast in cleanup. Path B writes one Parquet file with `pyarrow.parquet.write_table` to `s3://{BUCKET}/orders/sample-LOAD00000001.parquet`. The `pymysql` package needs to be installed before the seed runs; the cell handles that lazily via `pip install --quiet pymysql`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for the bucket and the RDS instance (Path A is the only path with a `create_db_instance` call; Path B uses the `pyarrow` seed loop already in the cell):

```python
s3.create_bucket(Bucket=BUCKET_NAME)

rds.create_db_instance(
    DBInstanceIdentifier=RDS_ID,
    AllocatedStorage=20,
    DBInstanceClass=DB_INSTANCE_CLASS,
    Engine=DB_ENGINE,
    MasterUsername=DB_USER,
    MasterUserPassword=DB_PASSWORD,
    DBName=DB_NAME,
    VpcSecurityGroupIds=[RDS_SG_ID],
    PubliclyAccessible=True,
    BackupRetentionPeriod=0,
    Port=DB_PORT,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

If `create_db_instance` returns `AccessDenied`, the `labrun-test` IAM user is missing `AmazonRDSFullAccess`. If `pymysql.connect` raises `OperationalError: (2003, "Can't connect to MySQL server")`, the SG rule for the caller IP is missing or the RDS instance came up private; `rds.modify_db_instance(DBInstanceIdentifier=RDS_ID, PubliclyAccessible=True, ApplyImmediately=True)` flips the flag without recreating the instance. If the seed cell raises `pymysql.err.OperationalError: SSL connection error`, your AWS region or VPC blocks outbound 443 to RDS; retry from a different network.

</details>


**Wallet check.** In Path A the RDS `db.t4g.micro` started billing at `create_db_instance` time. db.t4g.micro is $0.017 per hour off the free tier and free during the first 12 months of an AWS account, so if you said `y` to the free-tier prompt the meter reads $0.00. In Path B no critical resources have started; you have spent a few S3 PUT pennies and nothing else. The DMS replication instance comes up in Task 2 and that meter ($0.018/hour) bills regardless of free-tier status; that is the meter the 2-hour safety net is for.


## Task 2: Build the DMS replication instance, the safety net, the source endpoint, and the S3 target endpoint

Four pieces of plumbing land in this task:

1. **DMS subnet group.** Two subnets from the default VPC. DMS will not provision into a single-subnet group even on single-AZ instances.
2. **DMS IAM role for the S3 target.** Trust policy lets `dms.amazonaws.com` assume; permissions cover write on the lab bucket.
3. **Safety-net pair.** EventBridge schedule rule plus Lambda that, at the 2-hour mark, tag-checks the three critical resources (`dms_replication_task`, `dms_replication_instance`, `rds_instance`) and deletes any that still carry `labrun:lab-slug == "dms-database-migration"`. The Lambda body uses the execution role and does not pass `aws_session_token` (this is the one notebook-level exception to the session-token rule, documented in the cell).
4. **DMS replication instance.** `dms.t3.micro`, single-AZ, in the subnet group with the DMS SG attached. **The meter starts at `create_replication_instance` time and bills until `delete_replication_instance` completes.**

Once the instance is `available`, two endpoints come up: a MySQL source endpoint pointed at the RDS instance (Path A) or skipped (Path B), and an S3 target endpoint pointed at `s3://{BUCKET}/orders/` writing Parquet. The source endpoint connection uses SSL mode `require` because RDS forces TLS on newer engines. Both endpoints are tested with `dms.test_connection` and `describe_connections` until each reports `successful`.

### Checkpoint 2 validates

- DMS replication instance `available`, `dms.describe_endpoints` returns the source and target endpoints, and `dms.test_connection` reports `successful` for both within 60 seconds of polling.
- In Path B (USE_RDS == False), the source endpoint is created against a stub MySQL host (the RDS endpoint that does not exist); `test_connection` is expected to fail on the source, so Checkpoint 2 short-circuits to pass on the target endpoint only and notes the sample-data path was taken.


In [ ]:
# NBVAL_SKIP
# Task 2: DMS subnet group, S3 target role, safety-net pair (EventBridge rule
# + Lambda), DMS replication instance, source endpoint, target endpoint.
# The 2-hour safety net is created BEFORE the DMS replication instance so
# the seatbelt is in place when the meter starts.

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
dms = boto3.client(
    "dms",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
events = boto3.client(
    "events",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
lambda_client = boto3.client(
    "lambda",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# 1. DMS subnet group. Needs >= 2 subnets even on single-AZ.
# YOUR CODE: dms.create_replication_subnet_group(
#     ReplicationSubnetGroupIdentifier=SUBNET_GROUP_NAME,
#     ReplicationSubnetGroupDescription=f"Subnet group for {LAB_ID}",
#     SubnetIds=SUBNET_IDS,
#     Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
print(f"DMS subnet group created: {SUBNET_GROUP_NAME}")

# 2. DMS S3 target role. Trust policy lets dms.amazonaws.com assume.
# Inline policy uses s3:* wildcard which the IAM-policy validator accepts
# alongside literal required actions per LAB-CREATION-BLUEPRINT.
dms_s3_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "dms.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
dms_s3_inline_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": "s3:*",
        "Resource": [
            f"arn:aws:s3:::{BUCKET_NAME}",
            f"arn:aws:s3:::{BUCKET_NAME}/*",
        ],
    }],
}
# YOUR CODE: iam.create_role(
#     RoleName=DMS_S3_ROLE_NAME,
#     AssumeRolePolicyDocument=json.dumps(dms_s3_trust_policy),
#     Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# YOUR CODE: iam.put_role_policy(
#     RoleName=DMS_S3_ROLE_NAME,
#     PolicyName="labrun-dms-s3-policy",
#     PolicyDocument=json.dumps(dms_s3_inline_policy),
# ).
dms_s3_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{DMS_S3_ROLE_NAME}"
print(f"DMS S3 role ARN: {dms_s3_role_arn}")

# 3. Safety-net IAM role + Lambda + EventBridge rule.
sn_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
sn_inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "dms:*",
            "Resource": "*",
        },
        {
            "Effect": "Allow",
            "Action": [
                "rds:DescribeDBInstances",
                "rds:DeleteDBInstance",
                "rds:ListTagsForResource",
            ],
            "Resource": "*",
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents",
            ],
            "Resource": "*",
        },
    ],
}
# YOUR CODE: iam.create_role(
#     RoleName=SAFETY_NET_ROLE_NAME,
#     AssumeRolePolicyDocument=json.dumps(sn_trust_policy),
#     Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# YOUR CODE: iam.put_role_policy(
#     RoleName=SAFETY_NET_ROLE_NAME,
#     PolicyName="labrun-safety-net-policy",
#     PolicyDocument=json.dumps(sn_inline_policy),
# ).
sn_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{SAFETY_NET_ROLE_NAME}"

# Safety-net Lambda source. Tag-scoped: each delete confirms
# labrun:lab-slug matches this lab before calling delete. boto3.client
# inside this source string uses the Lambda execution role and does not
# need aws_session_token (exception noted; the notebook-level rule does
# require session token on every boto3.client call).
lambda_source = f"""
import boto3

RDS_ID = "{RDS_ID}"
DMS_INSTANCE_ID = "{DMS_INSTANCE_ID}"
TASK_ID = "{TASK_ID}"
EXPECTED_TAG_KEY = "{LAB_TAG_KEY}"
EXPECTED_TAG_VALUE = "{LAB_TAG_VALUE}"


def _matches(tag_list):
    tags = {{t["Key"]: t["Value"] for t in (tag_list or [])}}
    return tags.get(EXPECTED_TAG_KEY) == EXPECTED_TAG_VALUE


def handler(event, context):
    # boto3.client below uses the Lambda execution role; no session token.
    dms = boto3.client("dms")
    rds = boto3.client("rds")
    results = {{}}

    # 1. Stop + delete the DMS task first (must come before instance delete).
    try:
        tasks = dms.describe_replication_tasks(
            Filters=[{{"Name": "replication-task-id", "Values": [TASK_ID]}}]
        )["ReplicationTasks"]
        if tasks:
            task = tasks[0]
            task_arn = task["ReplicationTaskArn"]
            tag_resp = dms.list_tags_for_resource(ResourceArn=task_arn)
            if _matches(tag_resp.get("TagList", [])):
                if task.get("Status") not in ("stopped", "failed", "ready"):
                    try:
                        dms.stop_replication_task(ReplicationTaskArn=task_arn)
                    except Exception:
                        pass
                try:
                    dms.delete_replication_task(ReplicationTaskArn=task_arn)
                    results["task"] = "deleted"
                except Exception as exc:
                    results["task"] = f"error: {{exc}}"
            else:
                results["task"] = "refused (tag mismatch)"
        else:
            results["task"] = "absent"
    except Exception as exc:
        results["task"] = f"error: {{exc}}"

    # 2. Delete the DMS replication instance (this is what stops the bill).
    try:
        instances = dms.describe_replication_instances(
            Filters=[{{"Name": "replication-instance-id", "Values": [DMS_INSTANCE_ID]}}]
        )["ReplicationInstances"]
        if instances:
            inst = instances[0]
            inst_arn = inst["ReplicationInstanceArn"]
            tag_resp = dms.list_tags_for_resource(ResourceArn=inst_arn)
            if _matches(tag_resp.get("TagList", [])):
                try:
                    dms.delete_replication_instance(ReplicationInstanceArn=inst_arn)
                    results["instance"] = "deleted"
                except Exception as exc:
                    results["instance"] = f"error: {{exc}}"
            else:
                results["instance"] = "refused (tag mismatch)"
        else:
            results["instance"] = "absent"
    except Exception as exc:
        results["instance"] = f"error: {{exc}}"

    # 3. Delete the RDS instance with SkipFinalSnapshot.
    try:
        resp = rds.describe_db_instances(DBInstanceIdentifier=RDS_ID)
        db = resp["DBInstances"][0]
        tag_resp = rds.list_tags_for_resource(ResourceName=db["DBInstanceArn"])
        if _matches(tag_resp.get("TagList", [])):
            try:
                rds.delete_db_instance(
                    DBInstanceIdentifier=RDS_ID,
                    SkipFinalSnapshot=True,
                    DeleteAutomatedBackups=True,
                )
                results["rds"] = "deleted"
            except Exception as exc:
                results["rds"] = f"error: {{exc}}"
        else:
            results["rds"] = "refused (tag mismatch)"
    except rds.exceptions.DBInstanceNotFoundFault:
        results["rds"] = "absent"
    except Exception as exc:
        results["rds"] = f"error: {{exc}}"

    print(results)
    return results
"""

zbuf = io.BytesIO()
with zipfile.ZipFile(zbuf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("index.py", lambda_source)
zip_bytes = zbuf.getvalue()

# IAM role propagation can take a few seconds before Lambda accepts it.
_lambda_deadline = time.time() + 60
while True:
    try:
        # YOUR CODE: lambda_client.create_function(
        #     FunctionName=SAFETY_NET_LAMBDA_NAME,
        #     Runtime="python3.12",
        #     Role=sn_role_arn,
        #     Handler="index.handler",
        #     Code={"ZipFile": zip_bytes},
        #     Timeout=120,
        #     Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
        # ).
        break
    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidParameterValueException" and time.time() < _lambda_deadline:
            time.sleep(5)
            continue
        raise
lambda_arn = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{SAFETY_NET_LAMBDA_NAME}"
print(f"Safety-net Lambda created: {lambda_arn}")

# EventBridge schedule rule: one-shot at lab_start + SAFETY_NET_HOURS.
fire_at = (datetime.now(timezone.utc) + timedelta(hours=SAFETY_NET_HOURS)).replace(microsecond=0)
cron_expr = (
    f"cron({fire_at.minute} {fire_at.hour} "
    f"{fire_at.day} {fire_at.month} ? {fire_at.year})"
)
# YOUR CODE: events.put_rule(
#     Name=SAFETY_NET_RULE_NAME,
#     ScheduleExpression=cron_expr,
#     State="ENABLED",
#     Description=f"Safety net for {LAB_ID}; auto-deletes DMS task, instance, RDS at +{SAFETY_NET_HOURS}h.",
#     Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# YOUR CODE: lambda_client.add_permission(
#     FunctionName=SAFETY_NET_LAMBDA_NAME,
#     StatementId="labrun-eventbridge-invoke",
#     Action="lambda:InvokeFunction",
#     Principal="events.amazonaws.com",
#     SourceArn=f"arn:aws:events:{REGION}:{ACCOUNT_ID}:rule/{SAFETY_NET_RULE_NAME}",
# ).
# YOUR CODE: events.put_targets(
#     Rule=SAFETY_NET_RULE_NAME,
#     Targets=[{"Id": "1", "Arn": lambda_arn}],
# ).
print(f"EventBridge rule {SAFETY_NET_RULE_NAME} scheduled at {cron_expr}")

# 4. Create the DMS replication instance. The manifest entry was pre-declared
# in cell 4 with critical=True, so atexit catches a kernel crash during the
# waiter even though the instance is not yet "available".
# YOUR CODE: dms.create_replication_instance(
#     ReplicationInstanceIdentifier=DMS_INSTANCE_ID,
#     ReplicationInstanceClass=DMS_INSTANCE_CLASS,
#     AllocatedStorage=DMS_ALLOCATED_STORAGE,
#     VpcSecurityGroupIds=[DMS_SG_ID],
#     ReplicationSubnetGroupIdentifier=SUBNET_GROUP_NAME,
#     MultiAZ=False,
#     EngineVersion=DMS_ENGINE_VERSION,
#     PubliclyAccessible=True,
#     Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
print(f"create_replication_instance returned for {DMS_INSTANCE_ID}; DMS meter started at $0.018/hour.")
print("Waiting for DMS instance to become available (8 to 12 minutes typical)...")

# Poll-based waiter (boto3 ships a waiter named replication_instance_available).
deadline = time.time() + 1200
while time.time() < deadline:
    resp = dms.describe_replication_instances(
        Filters=[{"Name": "replication-instance-id", "Values": [DMS_INSTANCE_ID]}],
    )
    insts = resp.get("ReplicationInstances", [])
    if insts:
        status_ = insts[0]["ReplicationInstanceStatus"]
        if status_ == "available":
            DMS_INSTANCE_ARN = insts[0]["ReplicationInstanceArn"]
            break
        elif status_ in ("failed", "deleted"):
            print(f"DMS instance reached terminal state {status_!r}; aborting Task 2.")
            raise SystemExit(1)
    time.sleep(30)
else:
    print("DMS instance did not reach 'available' within 20 minutes.")
    raise SystemExit(1)
print(f"DMS instance {DMS_INSTANCE_ID} is available. ARN: {DMS_INSTANCE_ARN}")

# 5. Source endpoint (MySQL). In Path B the host is a stub.
if USE_RDS:
    source_server_name = RDS_ENDPOINT_HOST
else:
    source_server_name = f"{RDS_ID}.example-stub.us-east-1.rds.amazonaws.com"

src_resp = dms.create_endpoint(
    EndpointIdentifier=SOURCE_ENDPOINT_ID,
    EndpointType="source",
    EngineName="mysql",
    Username=DB_USER,
    Password=DB_PASSWORD,
    ServerName=source_server_name,
    Port=DB_PORT,
    DatabaseName=DB_NAME,
    SslMode="require",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
SOURCE_ENDPOINT_ARN = src_resp["Endpoint"]["EndpointArn"]
print(f"Source endpoint created: {SOURCE_ENDPOINT_ARN}")

# 6. Target endpoint (S3 Parquet).
tgt_resp = dms.create_endpoint(
    EndpointIdentifier=TARGET_ENDPOINT_ID,
    EndpointType="target",
    EngineName="s3",
    S3Settings={
        "ServiceAccessRoleArn": dms_s3_role_arn,
        "BucketName": BUCKET_NAME,
        "BucketFolder": "",
        "DataFormat": "parquet",
        "CompressionType": "GZIP",
    },
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
TARGET_ENDPOINT_ARN = tgt_resp["Endpoint"]["EndpointArn"]
print(f"Target endpoint created: {TARGET_ENDPOINT_ARN}")

# 7. Run test_connection on both endpoints. Poll describe_connections until
# the source reports successful (or fail in Path B because the host stub
# does not resolve; that is expected and the checkpoint handles it).
for arn, label in ((SOURCE_ENDPOINT_ARN, "source"), (TARGET_ENDPOINT_ARN, "target")):
    try:
        dms.test_connection(
            ReplicationInstanceArn=DMS_INSTANCE_ARN,
            EndpointArn=arn,
        )
    except ClientError as e:
        # If a previous run already left the connection record, ignore.
        if e.response["Error"]["Code"] != "InvalidResourceStateFault":
            raise

deadline = time.time() + 120
done = {"source": False, "target": False}
while time.time() < deadline and not all(done.values()):
    conns = dms.describe_connections(
        Filters=[{"Name": "replication-instance-arn", "Values": [DMS_INSTANCE_ARN]}],
    ).get("Connections", [])
    for conn in conns:
        if conn["EndpointArn"] == SOURCE_ENDPOINT_ARN:
            status_ = conn["Status"]
            if status_ in ("successful", "failed"):
                done["source"] = True
                print(f"source test_connection -> {status_}")
        elif conn["EndpointArn"] == TARGET_ENDPOINT_ARN:
            status_ = conn["Status"]
            if status_ in ("successful", "failed"):
                done["target"] = True
                print(f"target test_connection -> {status_}")
    if not all(done.values()):
        time.sleep(5)


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: DMS replication instance is available, source + target
# endpoints exist, and test_connection reports successful for both. In
# Path B (USE_RDS == False) the source endpoint is expected to fail
# test_connection because the host stub does not resolve; the checkpoint
# accepts a target-successful only and notes the sample-data path.


def checkpoint_2(session):
    try:
        dms_client = boto3.client(
            "dms",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            inst_resp = dms_client.describe_replication_instances(
                Filters=[{"Name": "replication-instance-id", "Values": [DMS_INSTANCE_ID]}],
            )
        except ClientError as e:
            return CheckpointResult(status="error", error_reason=str(e))
        insts = inst_resp.get("ReplicationInstances", [])
        if not insts:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"DMS replication instance {DMS_INSTANCE_ID!r} not found. "
                    f"Run the Task 2 cell to create it."
                ),
            )
        status_ = insts[0]["ReplicationInstanceStatus"]
        if status_ != "available":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"DMS instance status is {status_!r}, expected 'available'. "
                    f"Wait for the create_replication_instance waiter."
                ),
            )

        ep_resp = dms_client.describe_endpoints(
            Filters=[{"Name": "endpoint-id", "Values": [SOURCE_ENDPOINT_ID, TARGET_ENDPOINT_ID]}],
        )
        endpoints = ep_resp.get("Endpoints", [])
        ep_ids = {e["EndpointIdentifier"] for e in endpoints}
        if SOURCE_ENDPOINT_ID not in ep_ids:
            return CheckpointResult(
                status="fail",
                error_reason=f"Source endpoint {SOURCE_ENDPOINT_ID!r} does not exist.",
            )
        if TARGET_ENDPOINT_ID not in ep_ids:
            return CheckpointResult(
                status="fail",
                error_reason=f"Target endpoint {TARGET_ENDPOINT_ID!r} does not exist.",
            )

        deadline = time.time() + 60
        statuses = {"source": None, "target": None}
        while time.time() < deadline and not all(statuses.values()):
            conns = dms_client.describe_connections(
                Filters=[{"Name": "replication-instance-arn", "Values": [DMS_INSTANCE_ARN]}],
            ).get("Connections", [])
            for conn in conns:
                if conn["EndpointArn"] == SOURCE_ENDPOINT_ARN:
                    statuses["source"] = conn["Status"]
                elif conn["EndpointArn"] == TARGET_ENDPOINT_ARN:
                    statuses["target"] = conn["Status"]
            if not all(statuses.values()):
                time.sleep(3)

        if USE_RDS:
            if statuses["source"] != "successful":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Source test_connection is {statuses['source']!r}, expected 'successful'. "
                        f"Check the RDS SG allows ingress from the DMS SG on 3306, "
                        f"and the source endpoint SslMode is 'require'."
                    ),
                )
            if statuses["target"] != "successful":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Target test_connection is {statuses['target']!r}, expected 'successful'. "
                        f"Check the DMS S3 role has s3:* on the lab bucket."
                    ),
                )
            return CheckpointResult(status="pass")
        else:
            if statuses["target"] != "successful":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Target test_connection is {statuses['target']!r}, expected 'successful'. "
                        f"Check the DMS S3 role and bucket. "
                        f"(Sample-data path: only the target endpoint is asserted.)"
                    ),
                )
            return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks four properties in order: DMS instance exists, `ReplicationInstanceStatus == "available"`, both endpoints exist via `describe_endpoints`, and `test_connection` plus `describe_connections` report `successful` for both within 60 seconds. The most common miss on the first run is the wrong SSL mode on the source endpoint; RDS forces TLS by default on newer MySQL engines and the source endpoint must specify `SslMode="require"` (or `verify-ca`). The second most common miss is forgetting the DMS SG entry in the RDS SG ingress: the RDS SG must allow port 3306 from the DMS SG (a `UserIdGroupPairs` entry, not a CIDR). In the sample-data path the source endpoint is expected to fail because the host is a stub; the checkpoint only asserts the target endpoint.

</details>

<details><summary>Hint 2 (direction)</summary>

Five API calls in this task. `dms.create_replication_subnet_group(ReplicationSubnetGroupIdentifier=SUBNET_GROUP_NAME, ReplicationSubnetGroupDescription=..., SubnetIds=SUBNET_IDS, Tags=...)`. `iam.create_role` plus `iam.put_role_policy` for `DMS_S3_ROLE_NAME`. Same pattern for `SAFETY_NET_ROLE_NAME`. `lambda_client.create_function(FunctionName=SAFETY_NET_LAMBDA_NAME, Runtime="python3.12", Role=sn_role_arn, Handler="index.handler", Code={"ZipFile": zip_bytes}, Timeout=120, Tags={LAB_TAG_KEY: LAB_TAG_VALUE})` inside the retry loop. `events.put_rule` plus `lambda_client.add_permission` plus `events.put_targets` wire the safety net. Finally `dms.create_replication_instance(ReplicationInstanceIdentifier=DMS_INSTANCE_ID, ReplicationInstanceClass=DMS_INSTANCE_CLASS, AllocatedStorage=DMS_ALLOCATED_STORAGE, VpcSecurityGroupIds=[DMS_SG_ID], ReplicationSubnetGroupIdentifier=SUBNET_GROUP_NAME, MultiAZ=False, EngineVersion=DMS_ENGINE_VERSION, PubliclyAccessible=True, Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])`. The endpoint creation calls (`dms.create_endpoint` for source and target) are already in the cell because they depend on the resolved DMS instance ARN.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
dms.create_replication_subnet_group(
    ReplicationSubnetGroupIdentifier=SUBNET_GROUP_NAME,
    ReplicationSubnetGroupDescription=f"Subnet group for {LAB_ID}",
    SubnetIds=SUBNET_IDS,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

iam.create_role(
    RoleName=DMS_S3_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(dms_s3_trust_policy),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=DMS_S3_ROLE_NAME,
    PolicyName="labrun-dms-s3-policy",
    PolicyDocument=json.dumps(dms_s3_inline_policy),
)

iam.create_role(
    RoleName=SAFETY_NET_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(sn_trust_policy),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=SAFETY_NET_ROLE_NAME,
    PolicyName="labrun-safety-net-policy",
    PolicyDocument=json.dumps(sn_inline_policy),
)

lambda_client.create_function(
    FunctionName=SAFETY_NET_LAMBDA_NAME,
    Runtime="python3.12",
    Role=sn_role_arn,
    Handler="index.handler",
    Code={"ZipFile": zip_bytes},
    Timeout=120,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

events.put_rule(
    Name=SAFETY_NET_RULE_NAME,
    ScheduleExpression=cron_expr,
    State="ENABLED",
    Description=f"Safety net for {LAB_ID}; auto-deletes DMS task, instance, RDS at +{SAFETY_NET_HOURS}h.",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
lambda_client.add_permission(
    FunctionName=SAFETY_NET_LAMBDA_NAME,
    StatementId="labrun-eventbridge-invoke",
    Action="lambda:InvokeFunction",
    Principal="events.amazonaws.com",
    SourceArn=f"arn:aws:events:{REGION}:{ACCOUNT_ID}:rule/{SAFETY_NET_RULE_NAME}",
)
events.put_targets(
    Rule=SAFETY_NET_RULE_NAME,
    Targets=[{"Id": "1", "Arn": lambda_arn}],
)

dms.create_replication_instance(
    ReplicationInstanceIdentifier=DMS_INSTANCE_ID,
    ReplicationInstanceClass=DMS_INSTANCE_CLASS,
    AllocatedStorage=DMS_ALLOCATED_STORAGE,
    VpcSecurityGroupIds=[DMS_SG_ID],
    ReplicationSubnetGroupIdentifier=SUBNET_GROUP_NAME,
    MultiAZ=False,
    EngineVersion=DMS_ENGINE_VERSION,
    PubliclyAccessible=True,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

If `create_replication_instance` returns `AccessDenied`, the `labrun-test` IAM user is missing `AmazonDMSFullAccess`. If the waiter raises after 20 minutes, the instance may have hit a quota; check the DMS console and re-run the waiter (not `create_replication_instance` again). If the source endpoint's `test_connection` reports `Test Endpoint failed: Application-Status: 1020912, Application-Message: Cannot connect to ODBC provider`, the issue is almost always the SG path between DMS and RDS; confirm `UserIdGroupPairs` references the DMS SG, not just a CIDR.

</details>


**Wallet check.** **DMS meter started.** The moment `create_replication_instance` returned, the `dms.t3.micro` started billing at $0.018 per hour. Eight to twelve minutes to reach `available` is about half a cent. The 2-hour EventBridge safety net is now in place: even if your kernel dies right now, the Lambda will tag-check each of the three critical resources (DMS task, DMS instance, RDS instance) and call the right delete API at the 2-hour mark, capping worst case at about $0.07 from the DMS meter (plus $0.04 from RDS off free tier). RDS, IAM roles, security groups, the Lambda, and the EventBridge rule are all free at this volume. Keep moving; Task 3 starts the task and Task 4 validates the row count.


## Task 3: Create the full-load task, start it, and wait for it to finish

One API call kicks it off: `dms.create_replication_task` with `MigrationType="full-load"`, the source ARN, the target ARN, the replication-instance ARN, and the table-mapping JSON. Then `dms.start_replication_task(ReplicationTaskArn=task_arn, StartReplicationTaskType="start-replication")` and poll `describe_replication_tasks` until `Status` is one of the terminal states.

The DMS table-mapping JSON is the most common shape mistake in this lab: it must use a `selection` rule with `object-locator` (`schema-name` + `table-name`) and the `rule-action` `include`. The cert wants you to know that `rule-id` and `rule-name` are both required even though they look redundant; dropping either yields a cryptic `BadRequestException` at create-task time. The mapping for this lab is in the module-level `TABLE_MAPPINGS` constant.

In Path B (USE_RDS == False) Task 3 short-circuits: the create-task call would fail because the source endpoint cannot reach a real MySQL host. The cell skips the DMS task entirely; Checkpoint 3 validates against the sample Parquet seeded in Task 1 instead.

### Checkpoint 3 validates

- `USE_RDS == True`: `dms.describe_replication_tasks` returns one task with `Status == "stopped"` AND `StopReason` containing `FULL_LOAD_ONLY_FINISHED` (or the equivalent load-complete terminal state); task statistics report non-zero `FullLoadRows`.
- `USE_RDS == False`: the sample Parquet under `s3://{BUCKET}/orders/` exists and is non-empty (sample-data path).


In [ ]:
# NBVAL_SKIP
# Task 3: Create the full-load task, start it, and poll until terminal.
# Skipped entirely in Path B because no real source MySQL exists.

dms = boto3.client(
    "dms",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

if USE_RDS:
    # YOUR CODE: task_resp = dms.create_replication_task(
    #     ReplicationTaskIdentifier=TASK_ID,
    #     SourceEndpointArn=SOURCE_ENDPOINT_ARN,
    #     TargetEndpointArn=TARGET_ENDPOINT_ARN,
    #     ReplicationInstanceArn=DMS_INSTANCE_ARN,
    #     MigrationType="full-load",
    #     TableMappings=json.dumps(TABLE_MAPPINGS),
    #     Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    # ).
    # TASK_ARN = task_resp["ReplicationTask"]["ReplicationTaskArn"].
    deadline = time.time() + 120
    while time.time() < deadline:
        tasks = dms.describe_replication_tasks(
            Filters=[{"Name": "replication-task-id", "Values": [TASK_ID]}],
        ).get("ReplicationTasks", [])
        if tasks:
            t = tasks[0]
            if t.get("Status") == "ready":
                TASK_ARN = t["ReplicationTaskArn"]
                break
        time.sleep(5)
    else:
        print(f"DMS task {TASK_ID} did not reach 'ready' within 2 minutes.")
        raise SystemExit(1)
    print(f"DMS task {TASK_ID} is ready. ARN: {TASK_ARN}")

    # Start the task. YOUR CODE:
    # dms.start_replication_task(
    #     ReplicationTaskArn=TASK_ARN,
    #     StartReplicationTaskType="start-replication",
    # ).
    print("Starting full-load task; this typically takes 1 to 3 minutes on the seeded dataset.")
    deadline = time.time() + 600
    terminal_statuses = {"stopped", "failed"}
    last_status = None
    while time.time() < deadline:
        tasks = dms.describe_replication_tasks(
            Filters=[{"Name": "replication-task-id", "Values": [TASK_ID]}],
        ).get("ReplicationTasks", [])
        if tasks:
            t = tasks[0]
            last_status = t.get("Status")
            if last_status in terminal_statuses:
                print(f"Task status: {last_status}; StopReason: {t.get('StopReason')!r}")
                break
        time.sleep(15)
    else:
        print(f"DMS task did not finish within 10 minutes; last status {last_status!r}.")
        raise SystemExit(1)
else:
    print("Sample-data path: skipping DMS task creation. Checkpoint 3 validates the seeded Parquet.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: DMS full-load task is stopped with a load-complete StopReason
# and task statistics report non-zero FullLoadRows (Path A), or the sample
# Parquet under s3://{BUCKET}/orders/ exists and is non-empty (Path B).


def checkpoint_3(session):
    try:
        if USE_RDS:
            dms_client = boto3.client(
                "dms",
                aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
                aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
                aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
                region_name=REGION,
            )
            try:
                resp = dms_client.describe_replication_tasks(
                    Filters=[{"Name": "replication-task-id", "Values": [TASK_ID]}],
                )
            except ClientError as e:
                return CheckpointResult(status="error", error_reason=str(e))
            tasks = resp.get("ReplicationTasks", [])
            if not tasks:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"DMS task {TASK_ID!r} not found. "
                        f"Run the Task 3 cell to create and start it."
                    ),
                )
            t = tasks[0]
            status_ = t.get("Status")
            if status_ != "stopped":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Task status is {status_!r}, expected 'stopped'. "
                        f"Wait for the full-load task to finish, then re-run."
                    ),
                )
            stop_reason = t.get("StopReason", "") or ""
            # DMS reports load completion as one of: "FULL_LOAD_ONLY_FINISHED",
            # "Stop Reason FULL_LOAD_ONLY_FINISHED", or "Stop Reason FULL_LOAD_COMPLETED".
            ok_markers = ("FULL_LOAD_ONLY_FINISHED", "FULL_LOAD_COMPLETED", "load complete")
            if not any(m.lower() in stop_reason.lower() for m in ok_markers):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"StopReason is {stop_reason!r}; expected one of {ok_markers}. "
                        f"The task may have hit an error mid-load; check the DMS task logs."
                    ),
                )

            stats = t.get("ReplicationTaskStats", {}) or {}
            full_load_rows = stats.get("FullLoadProgressPercent", 0)
            # Modern DMS reports per-table stats via describe_table_statistics.
            ts_resp = dms_client.describe_table_statistics(
                ReplicationTaskArn=TASK_ARN or t["ReplicationTaskArn"],
            )
            ts_rows = sum(s.get("FullLoadRows", 0) for s in ts_resp.get("TableStatistics", []))
            if ts_rows < 1 and full_load_rows < 1:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Task statistics show zero FullLoadRows. The task finished "
                        f"but moved no rows; check the table-mapping JSON includes "
                        f"the correct schema-name and table-name."
                    ),
                )
            return CheckpointResult(status="pass")
        else:
            s3_client = boto3.client(
                "s3",
                aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
                aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
                aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
                region_name=REGION,
            )
            listing = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix="orders/")
            contents = listing.get("Contents", [])
            if not contents:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"No Parquet objects under s3://{BUCKET_NAME}/orders/. "
                        f"Re-run the Task 1 sample-data seed."
                    ),
                )
            total_bytes = sum(o.get("Size", 0) for o in contents)
            if total_bytes < 1:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Parquet objects exist but total size is {total_bytes} bytes.",
                )
            return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks three things in Path A: task exists, `Status == "stopped"`, and `StopReason` mentions full-load completion (`FULL_LOAD_ONLY_FINISHED` or `FULL_LOAD_COMPLETED`). Per-table statistics from `describe_table_statistics` must show non-zero `FullLoadRows`. The most common miss is the table-mapping JSON: the `selection` rule needs `rule-id`, `rule-name`, `object-locator` (with both `schema-name` and `table-name`), and `rule-action: include`. Drop any of those and `create_replication_task` returns `BadRequestException` at create time. In Path B the checkpoint validates the sample Parquet under `s3://{BUCKET}/orders/`.

</details>

<details><summary>Hint 2 (direction)</summary>

Two API calls. `dms.create_replication_task(ReplicationTaskIdentifier=TASK_ID, SourceEndpointArn=SOURCE_ENDPOINT_ARN, TargetEndpointArn=TARGET_ENDPOINT_ARN, ReplicationInstanceArn=DMS_INSTANCE_ARN, MigrationType="full-load", TableMappings=json.dumps(TABLE_MAPPINGS), Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])`. The task reaches `ready` within about 60 seconds; the poll loop in the cell handles that. Then `dms.start_replication_task(ReplicationTaskArn=TASK_ARN, StartReplicationTaskType="start-replication")`. The polling loop after that watches for `Status` in `("stopped", "failed")`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3:

```python
task_resp = dms.create_replication_task(
    ReplicationTaskIdentifier=TASK_ID,
    SourceEndpointArn=SOURCE_ENDPOINT_ARN,
    TargetEndpointArn=TARGET_ENDPOINT_ARN,
    ReplicationInstanceArn=DMS_INSTANCE_ARN,
    MigrationType="full-load",
    TableMappings=json.dumps(TABLE_MAPPINGS),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
TASK_ARN = task_resp["ReplicationTask"]["ReplicationTaskArn"]

# Wait for the task to reach "ready" (handled in the cell), then start it:
dms.start_replication_task(
    ReplicationTaskArn=TASK_ARN,
    StartReplicationTaskType="start-replication",
)
```

If `create_replication_task` returns `BadRequestException: Invalid JSON in TableMappings`, your `TABLE_MAPPINGS` dict is missing one of `rule-id`, `rule-name`, `object-locator.schema-name`, `object-locator.table-name`, or `rule-action`. If the task reaches `Status == "failed"` instead of `stopped`, the source endpoint cannot read from MySQL; revisit Checkpoint 2's source `test_connection` and the SG ingress on RDS. If `Status` stays `starting` for over 5 minutes, the DMS instance is still warming up its log capture; wait it out, the timer is 10 minutes total in the poll loop.

</details>


**Wallet check.** The DMS meter is still running at $0.018 per hour. The full-load task itself does not add a separate line item; you pay for the instance, not the task. The S3 PUT requests for the Parquet output are about $0.005 per 1000 PUTs, so a 10000-row migration writing one Parquet file lands well under a penny. Total session burn so far is about 1 to 3 cents (DMS meter for 8 to 15 minutes of `available` time, plus a few S3 PUTs). The big remaining variable is how long the DMS instance stays alive before cleanup. Move to Task 4 and the validation cell; then run cleanup immediately.


## Task 4: Validate the migration with a row-count assertion

The cert-scenario question this lab maps to is "prove every row made it across." The audit pattern is the same for any database-to-data-lake migration: count rows on the source, count rows in the target, assert equality (or near-equality, allowing a tiny delta for in-flight rows on CDC tasks).

For this lab the source count is `SELECT count(*) FROM orders` against the RDS MySQL (Path A) or the constant `SAMPLE_ROWS_EXPECTED` (Path B). The target count is the sum of `num_rows` across every Parquet file under `s3://{BUCKET}/orders/`. `pyarrow.parquet.read_metadata(file).num_rows` reads only the file footer, not the data, so the count is fast even on multi-gigabyte Parquet sets.

DMS full-load writes one or more Parquet files under the bucket folder (the cell's S3 target endpoint uses the bucket root as the prefix and the schema name as the folder). The validation cell paginates `list_objects_v2`, downloads each Parquet to a BytesIO, reads metadata, and sums. The result lands in `PARQUET_ROW_COUNT`. Checkpoint 4 reads `SOURCE_ROW_COUNT` and `PARQUET_ROW_COUNT` and asserts they are within plus or minus 1.

### Checkpoint 4 validates

- Both `SOURCE_ROW_COUNT` and `PARQUET_ROW_COUNT` are populated (non-None).
- `abs(SOURCE_ROW_COUNT - PARQUET_ROW_COUNT) <= 1`. In Path A, both come from live state. In Path B, `SOURCE_ROW_COUNT` is the constant `SAMPLE_ROWS_EXPECTED` and `PARQUET_ROW_COUNT` is the sum of the seeded sample Parquet.


In [ ]:
# NBVAL_SKIP
# Task 4: Read SOURCE_ROW_COUNT from MySQL (Path A) or the sample constant
# (Path B), enumerate Parquet files under s3://{BUCKET}/orders/, read
# pyarrow metadata to sum num_rows, and cache PARQUET_ROW_COUNT for the
# checkpoint to read.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

try:
    import pyarrow.parquet as pq  # noqa: F401
except ImportError:
    print("pyarrow not installed; installing now (about 15 seconds)...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "pyarrow"])
import pyarrow.parquet as pq

if USE_RDS:
    try:
        import pymysql
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "pymysql"])
        import pymysql
    desc = boto3.client(
        "rds",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    ).describe_db_instances(DBInstanceIdentifier=RDS_ID)
    host = desc["DBInstances"][0]["Endpoint"]["Address"]
    conn = pymysql.connect(
        host=host, port=DB_PORT,
        user=DB_USER, password=DB_PASSWORD,
        database=DB_NAME, connect_timeout=15,
        ssl={"ssl": {}},
    )
    try:
        with conn.cursor() as cur:
            # YOUR CODE: cur.execute("SELECT count(*) FROM orders;") then
            # SOURCE_ROW_COUNT = int(cur.fetchone()[0]).
            cur.execute("SELECT count(*) FROM orders;")
            SOURCE_ROW_COUNT = int(cur.fetchone()[0])
    finally:
        conn.close()
    print(f"SOURCE_ROW_COUNT (MySQL orders): {SOURCE_ROW_COUNT}")
else:
    SOURCE_ROW_COUNT = SAMPLE_ROWS_EXPECTED
    print(f"SOURCE_ROW_COUNT (sample constant): {SOURCE_ROW_COUNT}")

# Sum num_rows across every Parquet file under s3://{BUCKET}/orders/.
paginator = s3.get_paginator("list_objects_v2")
keys: list[str] = []
for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix="orders/"):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key.endswith(".parquet"):
            keys.append(key)

if not keys:
    print(f"No Parquet files found under s3://{BUCKET_NAME}/orders/. Task 3 may not have completed.")
    raise SystemExit(1)

PARQUET_ROW_COUNT = 0
for key in keys:
    body = s3.get_object(Bucket=BUCKET_NAME, Key=key)["Body"].read()
    buf = io.BytesIO(body)
    meta = pq.read_metadata(buf)
    PARQUET_ROW_COUNT += meta.num_rows
print(f"PARQUET_ROW_COUNT (sum across {len(keys)} files): {PARQUET_ROW_COUNT}")
print(f"Delta: {SOURCE_ROW_COUNT - PARQUET_ROW_COUNT}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: SOURCE_ROW_COUNT and PARQUET_ROW_COUNT are populated and
# within plus or minus 1 of each other. DMS full-load is exact in practice;
# the plus or minus 1 tolerance is there in case of a metadata read race.


def checkpoint_4(session):
    try:
        if SOURCE_ROW_COUNT is None or PARQUET_ROW_COUNT is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "SOURCE_ROW_COUNT or PARQUET_ROW_COUNT is None. "
                    "Run the Task 4 cell first."
                ),
            )
        delta = abs(int(SOURCE_ROW_COUNT) - int(PARQUET_ROW_COUNT))
        if delta > 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Row-count mismatch: source={SOURCE_ROW_COUNT}, "
                    f"parquet={PARQUET_ROW_COUNT}, delta={delta} (allowed <= 1). "
                    f"DMS full-load is exact; a delta > 1 means the task did not "
                    f"finish a full load, or the table-mapping JSON dropped rows."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint reads `SOURCE_ROW_COUNT` and `PARQUET_ROW_COUNT`, asserts both are populated, and asserts their absolute difference is at most 1. Both must be populated by Task 4 before the checkpoint runs; if you re-run Checkpoint 4 in a fresh kernel without running Task 4 first, the values are `None` and the checkpoint fails fast with that message. The most common miss when the delta is greater than 1 is that the DMS task did not actually finish a full load (Status is `running` not `stopped`); the second most common miss is the table-mapping JSON's `object-locator.table-name` not matching the actual MySQL table name exactly (case-sensitive on Linux MySQL).

</details>

<details><summary>Hint 2 (direction)</summary>

No new API calls beyond what Task 4 already does. The cell already paginates `s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix="orders/")`, filters to `.parquet` keys, downloads each via `s3.get_object`, and reads metadata via `pyarrow.parquet.read_metadata(BytesIO(body)).num_rows`. The MySQL count uses the same `pymysql.connect` path Task 1 used. Both totals land in module-level globals so Checkpoint 4 reads them without re-running anything.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the conceptual recipe; the cell already implements it:

```python
# 1. Source count from MySQL (Path A) or the sample constant (Path B).
if USE_RDS:
    conn = pymysql.connect(host=..., port=DB_PORT, user=DB_USER, password=DB_PASSWORD, database=DB_NAME, ssl={"ssl": {}})
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM orders;")
        SOURCE_ROW_COUNT = int(cur.fetchone()[0])
    conn.close()
else:
    SOURCE_ROW_COUNT = SAMPLE_ROWS_EXPECTED

# 2. Parquet count via pyarrow metadata.
PARQUET_ROW_COUNT = 0
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET_NAME, Prefix="orders/"):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            body = s3.get_object(Bucket=BUCKET_NAME, Key=obj["Key"])["Body"].read()
            PARQUET_ROW_COUNT += pq.read_metadata(io.BytesIO(body)).num_rows
```

If `PARQUET_ROW_COUNT` is zero but Task 3 reported `stopped`, the DMS S3 target wrote to a prefix you are not enumerating; revisit the S3 target endpoint's `BucketFolder` setting (this lab uses `BucketFolder=""` so DMS writes under `s3://{BUCKET}/{schema-name}/{table-name}/` which on a default config maps to `orders/`). If `SOURCE_ROW_COUNT` does not match the seeded count, the seed loop did not commit; re-run the seed inside Task 1.

</details>


**Wallet check.** Task 4 is bytes-cheap: a few S3 GETs to pull Parquet file footers (only the last 32 KB of each file is read by `pyarrow.parquet.read_metadata`, but boto3 downloads the whole object; for a 10000-row Parquet that is under 1 MB). The MySQL count is one query against the existing RDS connection. Total damage from Task 4: less than $0.001. The cluster meter (DMS instance plus RDS) is still running; the next cell runs cleanup which is what actually stops the bill. Do not pause here; go straight to cleanup.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation,
# critical-first order per RESOURCE-SAFETY-SPEC Rule 2 + Section 3 Lab 9.
#
# labrun-checks 0.3.0 lacks adapters for dms_replication_task,
# dms_replication_instance, rds_instance, dms_endpoint, dms_subnet_group,
# eventbridge_rule, lambda_function, and security_group; inline _LabAdapter
# adds them. When the package promotes these adapters in a future release,
# the wrapper can be removed and run_cleanup called against the default
# adapter.
#
# The DMS task and DMS instance adapters wait for the resource to actually
# be gone (or not-found) before returning so a second cleanup invocation
# (atexit racing the explicit run) does not double-delete or block. The
# RDS adapter passes SkipFinalSnapshot=True and DeleteAutomatedBackups=True.

from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter:
    """AwsCleanupAdapter wrapper that adds Lab 09 inline resource types.

    Inlined types: dms_replication_task, dms_replication_instance,
    rds_instance, dms_endpoint, dms_subnet_group, eventbridge_rule,
    lambda_function, security_group. Everything else (s3_bucket, iam_role)
    delegates to the bundled AwsCleanupAdapter unchanged.
    """

    def __init__(self) -> None:
        self._aws = AwsCleanupAdapter()

    def _client(self, service: str, credentials: dict):
        return boto3.client(
            service,
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
            aws_session_token=credentials.get("aws_session_token"),
            region_name=credentials.get("region", REGION),
        )

    def delete_resource(self, credentials: dict, resource) -> None:
        if resource.type == "dms_replication_task":
            client = self._client("dms", credentials)
            # Resolve ARN by id; manifest stores id for readability.
            tasks = client.describe_replication_tasks(
                Filters=[{"Name": "replication-task-id", "Values": [resource.id]}],
            ).get("ReplicationTasks", [])
            if not tasks:
                return
            arn = tasks[0]["ReplicationTaskArn"]
            status_ = tasks[0].get("Status")
            if status_ not in ("stopped", "failed", "ready", "creating"):
                try:
                    client.stop_replication_task(ReplicationTaskArn=arn)
                except ClientError as e:
                    if e.response["Error"]["Code"] not in ("InvalidResourceStateFault", "ResourceNotFoundFault"):
                        raise
                # Wait until stopped or terminal.
                deadline = time.time() + 300
                while time.time() < deadline:
                    refresh = client.describe_replication_tasks(
                        Filters=[{"Name": "replication-task-id", "Values": [resource.id]}],
                    ).get("ReplicationTasks", [])
                    if not refresh or refresh[0].get("Status") in ("stopped", "failed", "ready"):
                        break
                    time.sleep(10)
            try:
                client.delete_replication_task(ReplicationTaskArn=arn)
            except ClientError as e:
                if e.response["Error"]["Code"] not in ("ResourceNotFoundFault",):
                    raise
            # Wait for it to actually disappear.
            deadline = time.time() + 300
            while time.time() < deadline:
                remaining = client.describe_replication_tasks(
                    Filters=[{"Name": "replication-task-id", "Values": [resource.id]}],
                ).get("ReplicationTasks", [])
                if not remaining:
                    return
                time.sleep(10)
            return

        if resource.type == "dms_replication_instance":
            client = self._client("dms", credentials)
            insts = client.describe_replication_instances(
                Filters=[{"Name": "replication-instance-id", "Values": [resource.id]}],
            ).get("ReplicationInstances", [])
            if not insts:
                return
            arn = insts[0]["ReplicationInstanceArn"]
            try:
                client.delete_replication_instance(ReplicationInstanceArn=arn)
            except ClientError as e:
                code_ = e.response["Error"]["Code"]
                if code_ in ("ResourceNotFoundFault",):
                    return
                if code_ == "InvalidResourceStateFault":
                    # Already deleting from a prior call.
                    pass
                else:
                    raise
            deadline = time.time() + 900
            while time.time() < deadline:
                remaining = client.describe_replication_instances(
                    Filters=[{"Name": "replication-instance-id", "Values": [resource.id]}],
                ).get("ReplicationInstances", [])
                if not remaining:
                    return
                time.sleep(15)
            return

        if resource.type == "rds_instance":
            client = self._client("rds", credentials)
            try:
                client.delete_db_instance(
                    DBInstanceIdentifier=resource.id,
                    SkipFinalSnapshot=True,
                    DeleteAutomatedBackups=True,
                )
            except ClientError as e:
                code_ = e.response["Error"]["Code"]
                if code_ in ("DBInstanceNotFound",):
                    return
                if code_ == "InvalidDBInstanceState":
                    pass  # already deleting
                else:
                    raise
            deadline = time.time() + 900
            while time.time() < deadline:
                try:
                    client.describe_db_instances(DBInstanceIdentifier=resource.id)
                except ClientError as e:
                    if e.response["Error"]["Code"] == "DBInstanceNotFound":
                        return
                    raise
                time.sleep(15)
            return

        if resource.type == "dms_endpoint":
            client = self._client("dms", credentials)
            try:
                eps = client.describe_endpoints(
                    Filters=[{"Name": "endpoint-id", "Values": [resource.id]}],
                ).get("Endpoints", [])
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundFault":
                    return
                raise
            if not eps:
                return
            arn = eps[0]["EndpointArn"]
            try:
                client.delete_endpoint(EndpointArn=arn)
            except ClientError as e:
                if e.response["Error"]["Code"] not in ("ResourceNotFoundFault",):
                    raise
            return

        if resource.type == "dms_subnet_group":
            client = self._client("dms", credentials)
            try:
                client.delete_replication_subnet_group(
                    ReplicationSubnetGroupIdentifier=resource.id,
                )
            except ClientError as e:
                if e.response["Error"]["Code"] not in ("ResourceNotFoundFault",):
                    raise
            return

        if resource.type == "eventbridge_rule":
            client = self._client("events", credentials)
            try:
                client.remove_targets(Rule=resource.id, Ids=["1"], Force=True)
            except ClientError as e:
                if e.response["Error"]["Code"] not in (
                    "ResourceNotFoundException", "InternalException"
                ):
                    raise
            try:
                client.delete_rule(Name=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "ResourceNotFoundException":
                    raise
            return

        if resource.type == "lambda_function":
            client = self._client("lambda", credentials)
            try:
                client.delete_function(FunctionName=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "ResourceNotFoundException":
                    raise
            return

        if resource.type == "security_group":
            client = self._client("ec2", credentials)
            try:
                resp = client.describe_security_groups(
                    Filters=[{"Name": "group-name", "Values": [resource.id]}],
                )
                for sg in resp.get("SecurityGroups", []):
                    try:
                        client.delete_security_group(GroupId=sg["GroupId"])
                    except ClientError as e:
                        if e.response["Error"]["Code"] != "InvalidGroup.NotFound":
                            raise
            except ClientError as e:
                if e.response["Error"]["Code"] != "InvalidGroup.NotFound":
                    raise
            return

        return self._aws.delete_resource(credentials, resource)

    def describe_resource(self, credentials: dict, resource) -> bool:
        if resource.type == "dms_replication_task":
            client = self._client("dms", credentials)
            tasks = client.describe_replication_tasks(
                Filters=[{"Name": "replication-task-id", "Values": [resource.id]}],
            ).get("ReplicationTasks", [])
            return bool(tasks)

        if resource.type == "dms_replication_instance":
            client = self._client("dms", credentials)
            insts = client.describe_replication_instances(
                Filters=[{"Name": "replication-instance-id", "Values": [resource.id]}],
            ).get("ReplicationInstances", [])
            return bool(insts)

        if resource.type == "rds_instance":
            client = self._client("rds", credentials)
            try:
                client.describe_db_instances(DBInstanceIdentifier=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "DBInstanceNotFound":
                    return False
                raise

        if resource.type == "dms_endpoint":
            client = self._client("dms", credentials)
            try:
                eps = client.describe_endpoints(
                    Filters=[{"Name": "endpoint-id", "Values": [resource.id]}],
                ).get("Endpoints", [])
                return bool(eps)
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundFault":
                    return False
                raise

        if resource.type == "dms_subnet_group":
            client = self._client("dms", credentials)
            try:
                groups = client.describe_replication_subnet_groups(
                    Filters=[{"Name": "replication-subnet-group-id", "Values": [resource.id]}],
                ).get("ReplicationSubnetGroups", [])
                return bool(groups)
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundFault":
                    return False
                raise

        if resource.type == "eventbridge_rule":
            client = self._client("events", credentials)
            try:
                client.describe_rule(Name=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return False
                raise

        if resource.type == "lambda_function":
            client = self._client("lambda", credentials)
            try:
                client.get_function(FunctionName=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return False
                raise

        if resource.type == "security_group":
            client = self._client("ec2", credentials)
            try:
                resp = client.describe_security_groups(
                    Filters=[{"Name": "group-name", "Values": [resource.id]}],
                )
                return bool(resp.get("SecurityGroups", []))
            except ClientError as e:
                if e.response["Error"]["Code"] == "InvalidGroup.NotFound":
                    return False
                raise

        return self._aws.describe_resource(credentials, resource)

    def tag_scan(self, credentials: dict, lab_slug: str, region: str) -> list[str]:
        return self._aws.tag_scan(credentials, lab_slug, region)


# Empty the S3 bucket before run_cleanup tears it down.
print("Emptying the S3 bucket before teardown...")
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": keys})
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchBucket":
        print(f"Bucket emptying ran into: {e}. Continuing to cleanup.")

# Drop inline policies on the two roles before role-delete so the
# iam_role adapter does not block on attached policies.
iam_client = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
for role_name in (DMS_S3_ROLE_NAME, SAFETY_NET_ROLE_NAME):
    try:
        listed = iam_client.list_role_policies(RoleName=role_name)
        for policy_name in listed.get("PolicyNames", []):
            iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
    except ClientError as e:
        if e.response["Error"]["Code"] != "NoSuchEntity":
            print(f"Inline policy detach for {role_name} ran into: {e}. Continuing.")

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if getattr(r, "critical", False))
standard_total = len(CLEANUP_MANIFEST) - critical_total
critical_destroyed = critical_total - sum(
    1 for r in CLEANUP_MANIFEST if getattr(r, "critical", False) and r.id in failed_ids
)
standard_destroyed = standard_total - sum(
    1 for r in CLEANUP_MANIFEST if not getattr(r, "critical", False) and r.id in failed_ids
)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: $0.05 to $0.30.** The DMS replication instance is the only line item that materially registers. At $0.018 per hour, a 30-to-60-minute session of `available` time runs about 1 to 2 cents. RDS adds another 1 to 2 cents at $0.017 per hour if you are off free tier, $0.00 if you are within the first 12 months of your account. S3 PUT requests for the Parquet output land under $0.001. Worst case (kernel died, atexit ran late, the 2-hour EventBridge safety net fired) is closer to $0.30 because the DMS instance kept billing the full 2 hours. The catastrophic case (cleanup skipped, atexit failed, safety-net Lambda failed) would require three independent failures and is the reason the $20-per-month AWS billing alert exists. On a real production migration the DMS pricing pattern is the same: pay for the replication instance by the hour, not the data volume; size for parallelism and migration time, then delete the instance the moment the cutover finishes.


## These are not graded. They are for you.

Three questions worth sitting with before the exam.

1. You picked the `selection` rule type for the table mapping because you wanted exactly one table from one schema. Walk the alternatives. DMS supports `selection`, `transformation`, `table-settings`, and `post-processing` rule types. A `transformation` rule can rename a table on the fly (useful for case-conversion migrations between MySQL and Postgres); a `table-settings` rule can set parallel-load thresholds per table (useful when one table dominates the migration runtime); a `post-processing` rule can add tags to the S3 output (useful when downstream Glue crawlers key off tags). Identify two cases where each rule type is the clear winner: when the schema and table names match exactly across source and target, when the source name space conflicts with target conventions, when one table is 100 times larger than the rest, when the data lake's downstream crawlers need lifecycle tags.

2. The 2-hour EventBridge plus Lambda safety net cost about $0 to run and stopped a forgotten DMS replication instance from costing $3 over a week. The DMS-bills-while-stopped surprise is the canonical migration cost trap. Walk through the alternatives. AWS Budgets with a forecasted-spend action can shut down resources, but only at the account level (too blunt for a single lab). A CloudWatch alarm on `BillingEstimatedCharges` notifies but does not delete. A second EventBridge rule firing every 30 minutes could check the DMS instance's `InstanceCreateTime` and delete after 2 hours; this is actually simpler than a one-shot schedule and is what most production teams do for ephemeral migration instances. For each, identify when it is the right tool and when the one-shot safety net is the cleaner choice.

3. The row-count assertion was the only check at the end. Walk the alternatives. A full row-by-row hash comparison via `MD5(CONCAT(...))` on both sides catches data drift (rows that landed but with wrong values), but it costs a full table scan on both sides and quickly becomes prohibitive on multi-billion-row tables. Sampling 1000 random rows and comparing them by primary key is cheap but probabilistic; you accept some risk of missing a corrupted minority. AWS DMS itself supports `EnableValidation` which runs row-by-row hash comparisons in the background during CDC; on full-load-only tasks it is not enabled. Identify two cases where row-count is sufficient (migrating a small dimension table, migrating an append-only fact table within a single transactional source) and two where you need a stronger validator (migrating between engines with different character-set defaults, migrating a CDC stream where rows can mutate during the migration).

## Exam-Style Practice

**Q1.** A data engineer is migrating a 500 GB MySQL database to S3 using AWS DMS. The migration task is configured as `full-load` only. The team runs the task during off-hours and the cutover plan calls for the legacy database to be decommissioned 24 hours after the migration completes. Which cleanup sequence keeps DMS billing to a minimum?

A) Stop the replication task, then delete the replication instance, then delete the replication task, then delete the source and target endpoints.

B) Stop the replication task, then delete the replication task, then delete the replication instance, then delete the source and target endpoints.

C) Delete the replication instance first (this stops all billing), then delete the task, endpoints, and subnet group in any order.

D) Modify the replication instance to `dms.t3.nano` (the cheapest class) and leave it running so future migrations can reuse it.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. Deleting the replication instance before the task is deleted returns `InvalidResourceStateFault` because the task is still bound to the instance. The order is task-first, instance-second; both must be done explicitly because stopping a task does not free the instance, and the instance continues to bill while the task is stopped.
- B) Right. The canonical DMS cleanup sequence is stop-task, delete-task, delete-instance, delete-endpoints. The replication instance bills continuously from launch until deletion; stopping the task does not stop the bill. The endpoints and the subnet group are free at rest, so their order does not affect cost, only correctness (subnet group must come after both endpoints are gone).
- C) Wrong. `delete_replication_instance` returns `InvalidResourceStateFault` if a task is still bound to the instance, even if the task is in `stopped` state. The task must be deleted before the instance.
- D) Wrong. There is no `dms.t3.nano` class; the smallest is `dms.t3.micro`. Even if there were, leaving an instance running between migrations means paying for idle hours, which is exactly the trap the lab is teaching you to avoid. Spin up per migration, delete on cutover.

</details>

**Q2.** A team runs a DMS full-load task from RDS MySQL to S3 Parquet. The task completes with `Status == "stopped"` and `StopReason == "Stop Reason FULL_LOAD_ONLY_FINISHED"`, but a downstream Athena query against the Parquet returns 9999 rows when the source table has 10000. What is the most likely cause?

A) DMS dropped one row silently because the row had a NULL value in a non-nullable target column.

B) The DMS S3 target endpoint wrote one row to a different prefix than the one Athena is reading; check the `BucketFolder` and the schema-table path.

C) Athena is reading a partial Parquet file because the migration is still in progress; wait for the task to truly finish.

D) The source MySQL had 9999 rows at the moment DMS read it; the source got an extra insert after the migration started.

<details><summary>Show answer</summary>

**D**.

- A) Wrong. DMS would not drop a row silently; it would log a row-level error to the task statistics. `FullLoadRows` would be 9999 and `FullLoadErrorRows` would be 1, and `Status` would still be `stopped` with a different `StopReason`. The task statistics would not match the source either.
- B) Wrong. A misconfigured `BucketFolder` would split rows across two prefixes, not delete one row. The row count under the queried prefix would be much lower than 9999, not exactly 9999.
- C) Wrong. `Status == "stopped"` plus `StopReason == "FULL_LOAD_ONLY_FINISHED"` is a terminal state; the task is fully done. Parquet files written by DMS are complete at task-stop time.
- D) Right. DMS full-load reads a snapshot of the source at task-start time. Any DML against the source after the task started is not captured by a full-load-only task; you need `migration-type: full-load-and-cdc` or `cdc` to capture ongoing changes. The 1-row delta is almost always a new insert that landed after `start_replication_task` returned. This is also why the lab's row-count assertion uses a plus or minus 1 tolerance: it accepts exactly this case.

</details>

**Q3.** A team needs to migrate a 2 TB MySQL database to S3 Parquet with the smallest possible cutover window. The source database remains live during the migration and the team wants to minimize the row delta at cutover time. Which DMS migration type fits?

A) `full-load` only; run during a maintenance window with writes blocked.

B) `full-load-and-cdc`; the initial full load runs once, then CDC tracks ongoing changes until cutover.

C) `cdc` only; run for 24 hours then cut over.

D) `full-load` run twice back-to-back; the second run picks up any rows added during the first.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. Blocking writes on a live 2 TB database for the duration of a full-load migration (potentially hours) is exactly the cutover window the team is trying to minimize.
- B) Right. `full-load-and-cdc` is the canonical zero-downtime DMS migration pattern. The initial full load captures the snapshot; CDC reads the MySQL binlog and replays writes against the target continuously until the team flips the cutover switch. Cutover window is minutes, not hours, because CDC has kept the target nearly in sync.
- C) Wrong. `cdc` only does not capture the existing rows; it only reads the binlog from a starting position forward. You need a full-load first to seed the target with the existing data.
- D) Wrong. Running full-load twice does not solve the problem; the second run would re-read the entire 2 TB and any rows added between runs would still be missed for whatever interval between cutover and the second run's start. Also, DMS does not deduplicate rows on a re-run; you would get duplicates in the target.

</details>
